# SPATIAL INTELLIGENCE — PART 2
## Homework 02 — ETM

## 1. Import the needed libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\etmaglari\IAAC\etmaglari_gML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [2]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.29) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [3]:
renderer = "vscode"

## 4. Utility functions

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == 'face_id':
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan outline (OBJ)

In [5]:
OBJ_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework02\homework02_outline.obj"
objects = Topology.ByOBJPath(OBJ_PATH)
print(f"Imported {len(objects)} objects")

floor_face = None
for obj in objects:
    faces = Topology.Faces(obj)
    if faces:
        floor_face = faces[0]
        break
    wires = Topology.Wires(obj)
    if wires:
        floor_face = Face.ByWire(wires[0])
        if floor_face:
            break

print("Floor face loaded:", floor_face is not None)

b_r = Wire.BoundingRectangle(floor_face)
d_br = Topology.Dictionary(b_r)
xmin   = Dictionary.ValueAtKey(d_br, "xmin")
xmax   = Dictionary.ValueAtKey(d_br, "xmax")
ymin   = Dictionary.ValueAtKey(d_br, "ymin")
ymax   = Dictionary.ValueAtKey(d_br, "ymax")
width  = Dictionary.ValueAtKey(d_br, "width")
length = Dictionary.ValueAtKey(d_br, "length")
print(f"Bounds: x=[{xmin:.1f}, {xmax:.1f}]  y=[{ymin:.1f}, {ymax:.1f}]")
print(f"Size: {width:.1f} x {length:.1f} units")

Imported 2 objects
Floor face loaded: True
Bounds: x=[0.0, 30.5]  y=[-3.7, 12.2]
Size: 30.5 x 15.8 units


## 6. Show the floor plan

In [6]:
Topology.Show(floor_face,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 7. Create a grid overlay

In [9]:
GRID_STEP = 1
uRange = list(range(0, int(width)  + GRID_STEP, GRID_STEP))
vRange = list(range(0, int(length) + GRID_STEP, GRID_STEP))
grid = Grid.EdgesByDistances(floor_face, clip=True, uRange=uRange, vRange=vRange)

## 8. Show the floor plan and the grid

In [10]:
Topology.Show(floor_face, grid,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 9. Slice the floor plan with the grid to create a topologic shell

In [11]:
shell = Topology.Slice(floor_face, grid)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)
print(f"Grid cells: {len(faces)}")

Grid cells: 347


## 10. Derive the analysis graph from the shell

In [12]:
analysis_graph = Graph.ByTopology(shell)

## 11. Derive and store the graph vertices

In [13]:
g_verts = Graph.Vertices(analysis_graph)

## 12. Spatial Intelligence through Graph Analysis
### a. Community Detection

In [14]:
community_list = Graph.CommunityPartition(analysis_graph, colorScale="thermal")

In [15]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [16]:
Topology.Show(faces,
              faceColorKey="cp_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

### b. Degree Centrality
Bin faces by community, derive the outer boundary of each community, build a new shell and graph, then compute and interpolate degree centrality.

In [17]:
bins = Topology.BinByDictionaryKey(faces, key="community")
bin_dict = bins[0]
keys = list(bin_dict.keys())
face_groups = []
for key in keys:
    bin_faces = bin_dict[key]
    temp_shell = Shell.ByFaces(bin_faces)
    eb = Shell.ExternalBoundary(temp_shell)
    eb = Wire.RemoveCollinearEdges(eb)
    eb = Face.ByWire(eb)
    face_groups.append(eb)

In [18]:
Topology.Show(face_groups,
              faceOpacity=1,
              showEdges=True,
              edgeWidth=8,
              edgeColor="grey",
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

In [20]:
new_shell = Shell.ByFaces(face_groups)
new_graph = Graph.ByTopology(new_shell)
new_verts = Graph.Vertices(new_graph)
for v in new_verts:
    d = Dictionary.ByKeysValues(["color","size"],["red",12])
    v = Topology.SetDictionary(v, d)

In [21]:
Topology.Show(new_shell, new_graph,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              vertexSizeKey="size",
              vertexColorKey="color",
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

In [22]:
degree_centralities = Graph.DegreeCentrality(new_graph, normalize=False)

In [23]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="degree_centrality")

In [24]:
minValue = min(degree_centralities)
maxValue = max(degree_centralities)
for v in g_verts:
    d = Topology.Dictionary(v)
    d_c = Dictionary.ValueAtKey(d, "degree_centrality")
    color = Color.AnyToHex(Color.ByValueInRange(d_c, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "dc_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

In [25]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [26]:
Topology.Show(faces,
              faceColorKey="dc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              vertexSizeKey="size",
              vertexColorKey="dc_color",
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)